# Fourier analysis of Paul15 hematopoiesis (pseudotime)
This notebook adds **Fourier transform** analysis to the classic Paul et al. 2015 hematopoiesis dataset using Scanpy.

**Pipeline:** load data → neighbors → diffusion map → DPT pseudotime → uniform resample → rFFT → power spectrum → low‑pass iFFT reconstruction.

**References:**
- Scanpy Paul15 tutorial (paga‑paul15)
- Paul et al., Cell (2015) myeloid progenitors
- Scanpy (PyPI)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
sc.set_figure_params(dpi=110, figsize=(4.5,3.5), facecolor='white')
plt.rcParams['axes.grid'] = True
sc.settings.verbosity = 2
np.random.seed(0)
print(sc.__version__)


## Helper functions
We (i) resample expression along pseudotime to a **uniform grid** (needed for FFT), (ii) compute rFFT, and (iii) apply a simple low‑pass filter by zeroing high‑frequency coefficients before iFFT.


In [ ]:
def resample_by_pseudotime(pt, y, n_points=256):
    import numpy as np
    pt = np.asarray(pt).astype(float)
    y = np.asarray(y).astype(float)
    order = np.argsort(pt)
    pt_sorted, y_sorted = pt[order], y[order]
    t_grid = np.linspace(pt_sorted.min(), pt_sorted.max(), n_points)
    y_grid = np.interp(t_grid, pt_sorted, y_sorted)
    return t_grid, y_grid

def fft_power(y_grid):
    import numpy as np
    Y = np.fft.rfft(y_grid)
    power = (Y.real**2 + Y.imag**2)
    freqs = np.fft.rfftfreq(len(y_grid), d=1.0/len(y_grid))
    return freqs, Y, power

def low_pass_ifft(Y, cutoff_frac=0.2):
    import numpy as np
    Yf = Y.copy()
    k_cut = int(len(Yf) * cutoff_frac)
    Yf[k_cut:] = 0
    y_smooth = np.fft.irfft(Yf, n=(len(Yf)-1)*2)
    return y_smooth


## Load Paul15 data and compute DPT pseudotime
This mirrors the Scanpy tutorial: neighbors → diffusion map → DPT pseudotime.


In [ ]:
adata = sc.datasets.paul15()
# Per Scanpy notes: older versions returned log data; now returns non‑log counts.
sc.pp.log1p(adata)
sc.pp.neighbors(adata, n_neighbors=20, use_rep='X')
sc.tl.diffmap(adata)
sc.tl.dpt(adata, n_dcs=10)
pt = adata.obs['dpt_pseudotime'].to_numpy()
pt[:5], pt.shape


## Select genes and build trajectories
We use typical lineage markers **Gata1** (erythroid) and **Elane** (neutrophil)** if present; otherwise fall back to highly variable genes.


In [ ]:
candidate_genes = ['Gata1', 'Elane']
present = [g for g in candidate_genes if g in adata.var_names]
if len(present) < 2:
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
    fallback = list(adata.var[adata.var['highly_variable']].index[:2])
    genes = (present + fallback)[:2]
else:
    genes = present[:2]
genes


## Fourier analysis: raw vs low‑pass reconstructed trajectories
- For each gene: resample to uniform pseudotime grid → rFFT → power spectrum → iFFT after low‑pass.


In [ ]:
def expression_vs_pt(adata, gene):
    idx = np.where(adata.var_names == gene)[0]
    if len(idx) == 0:
        raise ValueError(f'Gene {gene} not found')
    x = np.asarray(adata.X[:, idx[0]]).reshape(-1)
    return adata.obs['dpt_pseudotime'].to_numpy(), x

for gene in genes:
    pt_raw, x_raw = expression_vs_pt(adata, gene)
    t_grid, y_grid = resample_by_pseudotime(pt_raw, x_raw, n_points=256)
    freqs, Y, power = fft_power(y_grid)
    y_lp = low_pass_ifft(Y, cutoff_frac=0.2)

    fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
    axs[0].scatter(pt_raw, x_raw, s=6, alpha=0.35)
    axs[0].set_title(f'{gene}: raw expression vs pseudotime')
    axs[0].set_xlabel('Pseudotime (DPT)'); axs[0].set_ylabel('log1p(expr)')

    axs[1].plot(t_grid, y_grid, lw=1.5, label='uniform resample')
    axs[1].plot(t_grid, y_lp, lw=2.0, label='low-pass iFFT (20%)')
    axs[1].set_title(f'{gene}: raw vs low-pass (time domain)')
    axs[1].set_xlabel('Pseudotime (resampled)'); axs[1].legend(loc='best')

    axs[2].plot(freqs[1:], power[1:], lw=1.5, color='tab:purple')
    axs[2].set_title(f'{gene}: power spectrum')
    axs[2].set_xlabel('Frequency bin'); axs[2].set_ylabel('|FFT|^2')
    axs[2].set_yscale('log')

    plt.tight_layout()
plt.show()


## Optional: branch‑specific spectra (example)
If your cluster labels (e.g., `adata.obs['paul15_clusters']`) map to erythroid vs neutrophil branches, compute spectra per branch and compare.


In [ ]:
print(sorted(adata.obs['paul15_clusters'].unique().tolist())[:10])
ery_like = ['MEP', 'Ery']  # edit to match your labels
neu_like = ['GMP', 'Neu']  # edit to match your labels
mask_ery = adata.obs['paul15_clusters'].isin(ery_like).to_numpy()
mask_neu = adata.obs['paul15_clusters'].isin(neu_like).to_numpy()

def fft_for_subset(gene, mask, n_points=256):
    pt_sub = adata.obs['dpt_pseudotime'].to_numpy()[mask]
    x_sub  = np.asarray(adata.X[mask, adata.var_names.get_loc(gene)]).reshape(-1)
    t_grid, y_grid = resample_by_pseudotime(pt_sub, x_sub, n_points=n_points)
    freqs, Y, power = fft_power(y_grid)
    return t_grid, y_grid, freqs, power

gene = genes[0]
t_e, y_e, f_e, p_e = fft_for_subset(gene, mask_ery)
t_n, y_n, f_n, p_n = fft_for_subset(gene, mask_neu)

fig, axs = plt.subplots(1,2, figsize=(9,3))
axs[0].plot(t_e, y_e, label=f'{gene} (ery-branch)')
axs[0].plot(t_n, y_n, label=f'{gene} (neu-branch)')
axs[0].set_title(f'{gene}: branch-wise trajectories'); axs[0].set_xlabel('Pseudotime (resampled)'); axs[0].legend()
axs[1].plot(f_e[1:], p_e[1:], label='ery', alpha=0.9)
axs[1].plot(f_n[1:], p_n[1:], label='neu', alpha=0.9)
axs[1].set_title(f'{gene}: power spectra by branch'); axs[1].set_xlabel('Frequency bin'); axs[1].set_ylabel('|FFT|^2'); axs[1].set_yscale('log'); axs[1].legend()
plt.tight_layout(); plt.show()
